In [3]:
# ---- CELDA 1: Evaluación de tracking — cargar datos ----
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "motmetrics", "--quiet"])

import json, csv, configparser
import numpy as np
from pathlib import Path
from collections import defaultdict

REPO_ROOT = Path("../..").resolve()
CLIP = "SNMOT-146"

# Tu salida
JSON_IN = REPO_ROOT / "data" / "tracking" / f"integrado_soccernet_{CLIP}.json"
with open(JSON_IN, encoding="utf-8") as fh:
    tu = json.load(fh)
tus_frames = tu["frames"]

# Ground truth
GT_FILE = REPO_ROOT / "data" / "soccernet" / "tracking" / "test" / CLIP / "gt" / "gt.txt"
gt_por_frame = defaultdict(list)
with open(GT_FILE) as fh:
    for fila in csv.reader(fh):
        fr, tid = int(fila[0]), int(fila[1])
        x, y, w, h = float(fila[2]), float(fila[3]), float(fila[4]), float(fila[5])
        gt_por_frame[fr].append((tid, x, y, w, h))

# Clase de cada track del GT (desde gameinfo)
INFO_FILE = REPO_ROOT / "data" / "soccernet" / "tracking" / "test" / CLIP / "gameinfo.ini"
cfg = configparser.ConfigParser(); cfg.read(INFO_FILE)
def cat_gt(desc):
    d = desc.lower()
    if "ball" in d: return "balon"
    if "referee" in d: return "arbitro"
    if "player" in d or "goalkeeper" in d: return "persona"
    return "otro"
clase_gt = {}
for key in cfg["Sequence"]:
    if key.startswith("trackletid_"):
        clase_gt[int(key.split("_")[1])] = cat_gt(cfg["Sequence"][key])

print(f"Tu JSON: {len(tus_frames)} frames")
print(f"GT: {sum(len(v) for v in gt_por_frame.values())} detecciones")
from collections import Counter
print(f"Clases GT: {dict(Counter(clase_gt.values()))}")

Tu JSON: 750 frames
GT: 7453 detecciones
Clases GT: {'persona': 19, 'arbitro': 2, 'balon': 1}


In [4]:
# ---- CELDA 2: MOT de personas (jugadores+porteros) ----
import motmetrics as mm

def iou_xywh(a, b):
    ax1,ay1,ax2,ay2 = a[0],a[1],a[0]+a[2],a[1]+a[3]
    bx1,by1,bx2,by2 = b[0],b[1],b[0]+b[2],b[1]+b[3]
    ix1,iy1,ix2,iy2 = max(ax1,bx1),max(ay1,by1),min(ax2,bx2),min(ay2,by2)
    if ix2<ix1 or iy2<iy1: return 0.0
    inter=(ix2-ix1)*(iy2-iy1)
    return inter/(a[2]*a[3]+b[2]*b[3]-inter)

acc = mm.MOTAccumulator(auto_id=True)

for i, tus_dets in enumerate(tus_frames):
    gt_frame = i + 1   # tu frame 0 = GT frame 1

    # GT: solo personas (jugador+portero)
    gt_ids, gt_boxes = [], []
    for (tid,x,y,w,h) in gt_por_frame.get(gt_frame, []):
        if clase_gt.get(tid) == "persona":
            gt_ids.append(tid); gt_boxes.append((x,y,w,h))

    # Tus tracks: jugador+portero
    tu_ids, tu_boxes = [], []
    for d in tus_dets:
        if d.get("rol") in ("jugador","portero"):
            x1,y1,x2,y2 = d["bbox"]
            tu_ids.append(d["id"]); tu_boxes.append((x1,y1,x2-x1,y2-y1))

    # Matriz de distancias (1-IoU); inf si IoU < 0.5
    if gt_boxes and tu_boxes:
        dist = np.zeros((len(gt_boxes), len(tu_boxes)))
        for gi, gb in enumerate(gt_boxes):
            for ti, tb in enumerate(tu_boxes):
                v = iou_xywh(gb, tb)
                dist[gi, ti] = 1 - v if v >= 0.5 else np.nan
    else:
        dist = np.empty((len(gt_boxes), len(tu_boxes)))
    acc.update(gt_ids, tu_ids, dist)

mh = mm.metrics.create()
resumen = mh.compute(acc, metrics=['mota','idf1','num_switches','mostly_tracked',
                                    'mostly_lost','num_false_positives','num_misses',
                                    'num_objects','num_unique_objects'], name='SNMOT-192')
print("=== MÉTRICAS MOT (jugadores+porteros) ===\n")
print(resumen.to_string())
print()
m = resumen.iloc[0]
print(f"MOTA: {m['mota']:.3f}  (1.0 = perfecto, combina FP+FN+switches)")
print(f"IDF1: {m['idf1']:.3f}  (mantenimiento de identidad)")
print(f"ID switches: {int(m['num_switches'])}  (cambios de identidad)")
print(f"Mostly Tracked: {int(m['mostly_tracked'])} / {int(m['num_unique_objects'])} objetos")
print(f"Mostly Lost: {int(m['mostly_lost'])}")

=== MÉTRICAS MOT (jugadores+porteros) ===

               mota      idf1  num_switches  mostly_tracked  mostly_lost  num_false_positives  num_misses  num_objects  num_unique_objects
SNMOT-192  0.739775  0.633547            39              11            1                  419        1139         6137                  19

MOTA: 0.740  (1.0 = perfecto, combina FP+FN+switches)
IDF1: 0.634  (mantenimiento de identidad)
ID switches: 39  (cambios de identidad)
Mostly Tracked: 11 / 19 objetos
Mostly Lost: 1


In [5]:
# ---- CELDA 3: Balón — cobertura y error de posición (no MOT) ----
def centro_xywh(b): return (b[0]+b[2]/2, b[1]+b[3]/2)

frames_gt_balon = 0      # frames donde el GT tiene balón
frames_tu_balon = 0      # frames donde tú detectas balón
ambos = 0                # frames donde ambos
errores = []             # distancia en píxeles cuando ambos

for i, tus_dets in enumerate(tus_frames):
    gt_frame = i + 1
    gt_balon = [(x,y,w,h) for (tid,x,y,w,h) in gt_por_frame.get(gt_frame, [])
                if clase_gt.get(tid) == "balon"]
    tu_balon = [d["bbox"] for d in tus_dets if d.get("rol") == "balon"]

    if gt_balon: frames_gt_balon += 1
    if tu_balon: frames_tu_balon += 1
    if gt_balon and tu_balon:
        ambos += 1
        cg = centro_xywh(gt_balon[0])
        x1,y1,x2,y2 = tu_balon[0]; ct = ((x1+x2)/2, (y1+y2)/2)
        errores.append(np.hypot(ct[0]-cg[0], ct[1]-cg[1]))

N = tu["n_frames"]
print("=== BALÓN (métrica propia) ===\n")
print(f"Frames con balón en GT: {frames_gt_balon}/{N} = {100*frames_gt_balon/N:.0f}%")
print(f"Frames donde detectas balón: {frames_tu_balon}/{N} = {100*frames_tu_balon/N:.0f}%")
print(f"Frames donde ambos tienen balón: {ambos}")
print(f"\nCobertura (de los frames con balón real, cuántos detectas):")
print(f"  {ambos}/{frames_gt_balon} = {100*ambos/max(frames_gt_balon,1):.0f}%")
if errores:
    errores = np.array(errores)
    print(f"\nError de posición del balón (píxeles, cuando ambos lo tienen):")
    print(f"  media: {errores.mean():.1f}px | mediana: {np.median(errores):.1f}px | p90: {np.percentile(errores,90):.1f}px")
    print(f"  dentro de 15px: {100*(errores<=15).mean():.0f}% | dentro de 30px: {100*(errores<=30).mean():.0f}%")

=== BALÓN (métrica propia) ===

Frames con balón en GT: 750/750 = 100%
Frames donde detectas balón: 361/750 = 48%
Frames donde ambos tienen balón: 361

Cobertura (de los frames con balón real, cuántos detectas):
  361/750 = 48%

Error de posición del balón (píxeles, cuando ambos lo tienen):
  media: 42.2px | mediana: 11.9px | p90: 18.9px
  dentro de 15px: 81% | dentro de 30px: 92%
